In [1]:
import numpy as np
from collections import defaultdict
from math import sqrt
from load import load_detected_complexes, load_reference_complexes
import os

# ======================== Cluster Quality Metrics ========================
def cohesiveness(cluster, graph):
    W_in = 0
    W_out = 0
    for node in cluster:
        for neighbor, weight in graph.get(node, {}).items():
            if neighbor in cluster:
                W_in += weight
            else:
                W_out += weight
    return W_in / (W_in + W_out) if (W_in + W_out) > 0 else 0

def density(cluster, graph):
    W_in = 0
    for node in cluster:
        for neighbor, weight in graph.get(node, {}).items():
            if neighbor in cluster:
                W_in += weight
    n = len(cluster)
    return (2 * W_in) / (n * (n - 1)) if n > 1 else 0

def AIEW(cluster, graph):
    W_in = 0
    E_C = 0
    for node in cluster:
        for neighbor, weight in graph.get(node, {}).items():
            if neighbor in cluster:
                W_in += weight
                E_C += 1
    return W_in / E_C if E_C > 0 else 0

def ABEW(cluster, graph):
    W_out = 0
    BE_C = 0
    for node in cluster:
        for neighbor, weight in graph.get(node, {}).items():
            if neighbor not in cluster:
                W_out += weight
                BE_C += 1
    return W_out / BE_C if BE_C > 0 else 0

def AWM(cluster, graph):
    aiew = AIEW(cluster, graph)
    abew = ABEW(cluster, graph)
    return aiew / (aiew + abew) if (aiew + abew) > 0 else 0

def FF(cluster, graph):
    return (
        cohesiveness(cluster, graph) +
        density(cluster, graph) +
        AIEW(cluster, graph) -
        ABEW(cluster, graph) +
        AWM(cluster, graph)
    )

def FS_fitness(individual, graph):
    return sum(FF(cluster, graph) for cluster in individual)

# ======================== Evaluation Metrics ========================
def overlap_score(pred, real):
    return len(set(pred) & set(real)) / sqrt(len(pred) * len(real)) if pred and real else 0

def jaccard_index(pred, real):
    return len(pred & real) / len(pred | real) if pred | real else 0

def compute_metrics(predicted_complexes, real_complexes, threshold=0.2):
    m = len(predicted_complexes)
    n = len(real_complexes)

    # PPV
    ppv_numerator = sum(max(len(set(pred) & set(real)) for real in real_complexes) for pred in predicted_complexes)
    ppv_denominator = sum(len(pred) for pred in predicted_complexes)
    ppv = ppv_numerator / ppv_denominator if ppv_denominator > 0 else 0

    # Sn
    sn_numerator = sum(max(len(set(pred) & set(real)) for pred in predicted_complexes) for real in real_complexes)
    sn_denominator = sum(len(real) for real in real_complexes)
    sn = sn_numerator / sn_denominator if sn_denominator > 0 else 0

    # F-measure
    f_measure = (2 * ppv * sn) / (ppv + sn) if (ppv + sn) > 0 else 0

    # Accuracy
    accuracy = sqrt(sn * ppv)

    # MMR
    mmr = np.mean([max(overlap_score(pred, real) for real in real_complexes) for pred in predicted_complexes]) if predicted_complexes else 0

    # Jaccard
    jaccard = np.mean([max(jaccard_index(set(pred), set(real)) for real in real_complexes) for pred in predicted_complexes]) if predicted_complexes else 0

    # Covered Rate
    matched_reals = sum(
        any(overlap_score(pred, real) >= threshold for pred in predicted_complexes)
        for real in real_complexes
    )
    covered_rate = matched_reals / n if n > 0 else 0

    return {
        "PPV": ppv,
        "Recall (Sn)": sn,
        "F-measure": f_measure,
        "Accuracy": accuracy,
        "MMR": mmr,
        "Jaccard": jaccard,
        "Covered Rate": covered_rate,
        "Score Total": f_measure + covered_rate + mmr + jaccard
    }

# ======================== Main Evaluation ========================
def evaluate_complexes(detected_path, reference_path, graph=None):
    # Load complexes
    detected_complexes = load_detected_complexes(detected_path)
    reference_complexes = load_reference_complexes(reference_path)
    
    # Convert to lists
    detected_list = list(detected_complexes.values())
    reference_list = list(reference_complexes.values())
    
    # Compute evaluation metrics
    metrics = compute_metrics(detected_list, reference_list)
    
    # Compute fitness scores if graph is provided
    if graph:
        metrics["FS_fitness"] = FS_fitness(detected_list, graph)
    
    return metrics

# ======================== Batch Processing ========================

def extract_network_name(filepath):
    """Extract network name from file path (e.g., 'BIOGRID_humain' from path)"""
    filename = os.path.basename(filepath)
    # Remove extension and any version numbers if needed
    network_name = os.path.splitext(filename)[0]
    return network_name

def batch_evaluate(file_pairs, output_file=None):
    results = []
    headers = [
        "Network",
        "Precision (PPV)",
        "Recall (Sn)",
        "F-measure",
        "Accuracy",
        "MMR",
        "Jaccard Index",
        "Covered Rate",
        "Total Score"
    ]
    
    for detected_path, reference_path in file_pairs:
        network_name = extract_network_name(reference_path)
        print(f"\nProcessing network: {network_name}")
        print(f"Detected: {os.path.basename(detected_path)}")
        print(f"Reference: {os.path.basename(reference_path)}")
        
        try:
            metrics = evaluate_complexes(detected_path, reference_path)
            row = {
                "Network": network_name,
                "Precision (PPV)": metrics["PPV"],
                "Recall (Sn)": metrics["Recall (Sn)"],
                "F-measure": metrics["F-measure"],
                "Accuracy": metrics["Accuracy"],
                "MMR": metrics["MMR"],
                "Jaccard Index": metrics["Jaccard"],
                "Covered Rate": metrics["Covered Rate"],
                "Total Score": metrics["Score Total"]
            }
            results.append(row)
            
            print("\nEvaluation Results:")
            for metric, value in row.items():
                if metric != "Network":
                    print(f"{metric}: {value:.4f}")
                
        except Exception as e:
            print(f"Error processing {network_name}: {str(e)}")
            results.append({
                "Network": network_name,
                "Error": str(e)
            })
    
    # Save results to TSV file
    if output_file:
        with open(output_file, 'w') as f:
            # Write header
            f.write("\t".join(headers) + "\n")
            
            # Write data rows
            for row in results:
                if "Error" in row:
                    continue  # Skip error rows or handle differently
                
                values = [
                    row["Network"],
                    f"{row['Precision (PPV)']:.4f}",
                    f"{row['Recall (Sn)']:.4f}",
                    f"{row['F-measure']:.4f}",
                    f"{row['Accuracy']:.4f}",
                    f"{row['MMR']:.4f}",
                    f"{row['Jaccard Index']:.4f}",
                    f"{row['Covered Rate']:.4f}",
                    f"{row['Total Score']:.4f}"
                ]
                f.write("\t".join(values) + "\n")
        
        print(f"\nResults saved to {output_file}")
    
    return results

# Configuration
FILE_PAIRS = [
    ("/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_BIOGRID_humain.txt",
     "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/BIOGRID_humain.txt"),
    ("/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_BIOGRID_levure.txt",
     "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/BIOGRID_levure.txt"),
    ("/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_DIP_levure.txt",
     "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/DIP_levure.txt"),
    ("/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_humain.txt",
     "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/STRING_humain.txt"),
    ("/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/complexes/other_mthods/COACH_complexes_STRING_levure.txt",
     "/Users/ryham/Documents/mémoireM2/Master_final_project/Data/clean data/complexes/STRING_levure.txt"),
    # Add your other 4 file pairs here in the same format
]

OUTPUT_TSV = "/Users/ryham/Documents/mémoireM2/Master_final_project/results/initialization/metrics/evaluation_results_coach.tsv"

if __name__ == "__main__":
    print("Starting batch evaluation of complex detection...")
    results = batch_evaluate(FILE_PAIRS, OUTPUT_TSV)
    
    # Print final summary
    print("\nFinal Results Summary:")
    print("Network\t\tPPV\tSn\tF1\tAccuracy\tMMR\tJaccard\tCovered\tTotal")
    for row in results:
        if "Error" not in row:
            print(f"{row['Network']}\t"
                  f"{row['Precision (PPV)']:.3f}\t"
                  f"{row['Recall (Sn)']:.3f}\t"
                  f"{row['F-measure']:.3f}\t"
                  f"{row['Accuracy']:.3f}\t"
                  f"{row['MMR']:.3f}\t"
                  f"{row['Jaccard Index']:.3f}\t"
                  f"{row['Covered Rate']:.3f}\t"
                  f"{row['Total Score']:.3f}")
    
    print("\nBatch evaluation completed!")

Starting batch evaluation of complex detection...

Processing network: BIOGRID_humain
Detected: COACH_complexes_BIOGRID_humain.txt
Reference: BIOGRID_humain.txt

Evaluation Results:
Precision (PPV): 0.0099
Recall (Sn): 0.9685
F-measure: 0.0196
Accuracy: 0.0980
MMR: 0.1396
Jaccard Index: 0.0763
Covered Rate: 0.0098
Total Score: 0.2454

Processing network: BIOGRID_levure
Detected: COACH_complexes_BIOGRID_levure.txt
Reference: BIOGRID_levure.txt

Evaluation Results:
Precision (PPV): 0.0168
Recall (Sn): 0.9885
F-measure: 0.0331
Accuracy: 0.1290
MMR: 0.1942
Jaccard Index: 0.1071
Covered Rate: 0.0081
Total Score: 0.3425

Processing network: DIP_levure
Detected: COACH_complexes_DIP_levure.txt
Reference: DIP_levure.txt

Evaluation Results:
Precision (PPV): 0.0107
Recall (Sn): 0.9881
F-measure: 0.0211
Accuracy: 0.1027
MMR: 0.2804
Jaccard Index: 0.2042
Covered Rate: 0.0202
Total Score: 0.5259

Processing network: STRING_humain
Detected: COACH_complexes_STRING_humain.txt
Reference: STRING_humain.